### SECTION 1 — Setup

In [2]:
# Imports and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

df = pd.read_csv("../data/nhs_waiting_times.csv")

# check data structure
# print(df.shape)
# print(df.columns[:20])

## Section 2 - Exploration of the dataset 

### 2.1 Dataset Overview

The dataset contains NHS Referral to Treatment (RTT) waiting time information, broken down by provider organisation, commissioning organisation, treatment specialty, and waiting-time bands.
The data is stored in a wide format, where each row represents waiting list counts for a provider, treatment type, and time period, broken down by waiting time bands.

The data frame consists of 121 columns including 105 bucket columns representing called "Gt A To B Weeks SUM 1" representing the number of patients whose waiting time for a consultant-led elective treatment is Greater than A weeks but B weeks or less. The remaining 16 columns include information on time period, provider, commissioner, and treatment function. 

In [4]:
df["Period"].unique()

array(['RTT-February-2026'], dtype=object)

The period column only contains RTT-February-2026, so it doesn't provide any useful information for analysis. We can drop it from the dataset.

### 2.2 Identify variable groups
The variables can broadly be divided into:
- Identifier variables (provider, commissioner, treatment function)
- Reporting variables (totals and unknown clock starts)
- Waiting-time distribution variables (weekly waiting buckets)

Some columns are repeats (for example, `Treatment function code` is equivalent to `Treatment function name`).

Each row represents waiting-time counts for a specific provider, commissioner, treatment specialty and RTT pathway category with patients distributed across weekly waiting-time bands.

In [ ]:
identifier_cols = [
    "Provider Parent Name",
    "Provider Org Name",
    "Commissioner Parent Name",
    "Commissioner Org Name",
    "Treatment Function Name",
    "RTT Part Description"
]

reporting_cols = [
    "Total",
    "Total All",
    "Patients with unknown clock start date"
]

wait_cols = [c for c in df.columns if "Weeks SUM 1" in c]

print("Identifier columns:", len(identifier_cols))
print("Reporting columns:", len(reporting_cols))
print("Waiting bucket columns:", len(wait_cols))

# Analyse an individual row
df.iloc[0]

Identifier columns: 6
Reporting columns: 3
Waiting bucket columns: 105


### 2.3 PROVIDER VS COMMISSIONER
The provider is the organisation delivering care, e.g. Leeds Teaching Hospitals NHS Trust. The commissioner is the organisation funding/commissioning care. We can identify the top providers and commissioners:

In [9]:
df["Provider Org Name"].value_counts().head()
df["Commissioner Org Name"].value_counts().head()

Commissioner Org Name
NHS NORTH WEST LONDON (SUB ICB LOCATION)    4055
NHS NORTH EAST LONDON (SUB ICB LOCATION)    3206
NHS SOUTH WEST LONDON (SUB ICB LOCATION)    3147
NHS KENT AND MEDWAY (SUB ICB LOCATION)      3090
NHS SURREY HEARTLANDS (SUB ICB LOCATION)    3065
Name: count, dtype: int64

### Section 3 - Data Quality Assessment

The column "Total Patients with unknown clock start date" represents the number of patients on an RTT pathway (waiting for treatment) where the trust cannot identify the precise date the RTT clock started, often due to administrative issues or delays in receiving referral information. This usually indicates a data quality issue or a failed Inter Provider Transfer (IPT). Hence, I will create a data quality metric quantifying this situation. 

df["unknown_rate"] = (
    df["Patients with unknown clock start date"] /
    df["Total All"]
)

We can use this metric to analyse which providers have the worst data quality and whether poor data quality correlate with long waits: